# Telco Customer Churn EDA
Exploratory data analysis for the IBM Telco Customer Churn dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


## Load dataset

In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## Check for missing values

In [ ]:
df.isnull().sum()

## Data Cleaning
- Partner, PaperlessBilling, and Churn converted to integers (1 for Yes, 0 for No).
- TotalCharges contained blanks for 11 customers with tenure 0 (less than a month); these rows were removed.

In [ ]:
yes_no_cols = ['Partner','PaperlessBilling','Churn']
le = LabelEncoder()
for col in yes_no_cols:
    df[col] = le.fit_transform(df[col])

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
rows_before = df.shape[0]
df = df.dropna(subset=['TotalCharges'])
rows_after = df.shape[0]
print(f'Dropped {rows_before-rows_after} rows due to missing TotalCharges')


## Exploratory Data Analysis

### Gender and Churn Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].pie(df['gender'].value_counts(), labels=df['gender'].value_counts().index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Gender Distribution')
axes[1].pie(df['Churn'].value_counts(), labels=['No','Yes'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Churn Distribution')
plt.show()


### Customer contract distribution

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title('Customer Contract Distribution by Churn')
plt.show()


### Payment method distribution

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(y='PaymentMethod', data=df, order=df['PaymentMethod'].value_counts().index)
plt.title('Payment Method Distribution')
plt.show()
plt.figure(figsize=(8,6))
sns.countplot(data=df, x='PaymentMethod', hue='Churn', order=df['PaymentMethod'].value_counts().index)
plt.xticks(rotation=45)
plt.title('Churn by Payment Method')
plt.show()


### Dependents and Partners

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
sns.countplot(data=df, x='Dependents', hue='Churn', ax=axes[0])
axes[0].set_title('Dependents Distribution by Churn')
sns.countplot(data=df, x='Partner', hue='Churn', ax=axes[1])
axes[1].set_title('Partner Distribution by Churn')
plt.tight_layout()
plt.show()


### Services and Churn

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(data=df, x='OnlineSecurity', hue='Churn')
plt.title('Churn w.r.t Online Security')
plt.show()
plt.figure(figsize=(8,6))
sns.countplot(data=df, x='PaperlessBilling', hue='Churn')
plt.title('Churn w.r.t Paperless Billing')
plt.show()
plt.figure(figsize=(8,6))
sns.countplot(data=df, x='TechSupport', hue='Churn')
plt.title('Churn w.r.t Tech Support')
plt.show()
plt.figure(figsize=(8,6))
sns.countplot(data=df, x='PhoneService', hue='Churn')
plt.title('Churn w.r.t Phone Service')
plt.show()


### Charges and Tenure

In [ ]:
sns.set_context('paper', font_scale=1.1)
ax = sns.kdeplot(df.MonthlyCharges[df['Churn'] == 0], color='Red', shade=True)
ax = sns.kdeplot(df.MonthlyCharges[df['Churn'] == 1], ax=ax, color='Blue', shade=True)
ax.legend(['Not Churn','Churn'],loc='upper right')
ax.set_ylabel('Density')
ax.set_xlabel('Monthly Charges')
ax.set_title('Distribution of monthly charges by churn')
plt.show()
ax = sns.kdeplot(df.TotalCharges[df['Churn'] == 0], color='Gold', shade=True)
ax = sns.kdeplot(df.TotalCharges[df['Churn'] == 1], ax=ax, color='Green', shade=True)
ax.legend(['Not Churn','Churn'],loc='upper right')
ax.set_ylabel('Density')
ax.set_xlabel('Total Charges')
ax.set_title('Distribution of total charges by churn')
plt.show()
plt.figure(figsize=(8,6))
sns.boxplot(x='Churn', y='tenure', data=df)
plt.title('Tenure vs Churn')
plt.xlabel('Churn')
plt.ylabel('Tenure (Months)')
plt.show()


### Correlation Heatmap

In [ ]:
plt.figure(figsize=(25, 10))
corr = df.apply(lambda x: pd.factorize(x)[0]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
ax = sns.heatmap(corr, mask=mask, xticklabels=corr.columns, yticklabels=corr.columns, annot=True, linewidths=.2, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

## Logistic Regression Model

In [ ]:
X = df.drop(columns=['Churn'])
X = pd.get_dummies(X, drop_first=True)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
preds = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, preds))
print(classification_report(y_test, preds))
